# Summarize

TODO: 
- We have strongly supposed that the '_' values where just zeros, that was a strong assupmtuion and we must treat them as null values
- There are plenty resource and methods to handle missing values, here we are applying just something similar to 'Last observation carried forward'
- The reason why is that we have indentified in these features () that the value does not change over the months, but it would be interesing to test how model changes with different approahces of handling misisng values

# Libraries

In [ ]:
import os

os.chdir("..")

import kagglehub
import pandas as pd
from kagglehub import KaggleDatasetAdapter

from src.data_cleaning import (
    cleaning_int_variables,
    clean_credit_history,
    clean_type_of_loan,
    fill_credit_age,
)

pd.set_option("display.max_columns", None)
pd.options.display.float_format = "{:,.2f}".format

/Users/luisfernandocorcueraleon/Desktop/code/Credit-Score-Classification/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/luisfernandocorcueraleon/Desktop/code/Credit-Score-Classification/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Data

In [ ]:
train_df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS, "parisrohan/credit-score-classification", "train.csv"
)

/var/folders/1f/54wv342x795c_zyzqlgyjjkw0000gn/T/ipykernel_1573/1138772737.py:1: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv('/Users/luisfernandocorcueraleon/Desktop/code/Credit-Score-Classification/data/train.csv')


In [9]:
month_mapping = {
    "January": 202201,
    "February": 202202,
    "March": 202203,
    "April": 202204,
    "May": 202205,
    "June": 202206,
    "July": 202207,
    "August": 202208,
}

train_df["num_month"] = train_df["Month"].map(month_mapping)

train_df["num_month"].value_counts()

num_month
202201    12500
202202    12500
202203    12500
202204    12500
202205    12500
202206    12500
202207    12500
202208    12500
Name: count, dtype: int64

# Cleaning Numerical Values:

In [10]:
train_df.columns

Index(['ID', 'Customer_ID', 'Month', 'Name', 'Age', 'SSN', 'Occupation',
       'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts',
       'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Type_of_Loan',
       'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit',
       'Num_Credit_Inquiries', 'Credit_Mix', 'Outstanding_Debt',
       'Credit_Utilization_Ratio', 'Credit_History_Age',
       'Payment_of_Min_Amount', 'Total_EMI_per_month',
       'Amount_invested_monthly', 'Payment_Behaviour', 'Monthly_Balance',
       'Credit_Score', 'num_month'],
      dtype='object')

There are many similar cases of numerical values that are object type and have '_' values to clean

In [11]:
train_df["Age"] = cleaning_int_variables(train_df["Age"], "_", "", int)
train_df["Annual_Income"] = cleaning_int_variables(
    train_df["Annual_Income"], "_", "", float
)
train_df["Num_of_Loan"] = cleaning_int_variables(train_df["Num_of_Loan"], "_", "", int)
train_df["Changed_Credit_Limit"] = cleaning_int_variables(
    train_df["Changed_Credit_Limit"], "_", "0", float
)
train_df["Outstanding_Debt"] = cleaning_int_variables(
    train_df["Outstanding_Debt"], "_", "", float
)
train_df["Amount_invested_monthly"] = cleaning_int_variables(
    train_df["Amount_invested_monthly"], "_", "", float
)
train_df["Monthly_Balance"] = cleaning_int_variables(
    train_df["Monthly_Balance"], "_", "", float
)
train_df["Num_of_Delayed_Payment"] = cleaning_int_variables(
    train_df["Num_of_Delayed_Payment"], "_", "", float
)

In [12]:
train_df["Credit_History_Age"] = train_df["Credit_History_Age"].apply(
    clean_credit_history
)
train_df["Credit_History_Age"].describe()

count   90,970.00
mean        18.43
std          8.31
min          0.08
25%         12.00
50%         18.25
75%         25.17
max         33.67
Name: Credit_History_Age, dtype: float64

In [13]:
numeric_df = train_df.select_dtypes(include="number")
numeric_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 18 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Age                       100000 non-null  int64  
 1   Annual_Income             100000 non-null  float64
 2   Monthly_Inhand_Salary     84998 non-null   float64
 3   Num_Bank_Accounts         100000 non-null  int64  
 4   Num_Credit_Card           100000 non-null  int64  
 5   Interest_Rate             100000 non-null  int64  
 6   Num_of_Loan               100000 non-null  int64  
 7   Delay_from_due_date       100000 non-null  int64  
 8   Num_of_Delayed_Payment    92998 non-null   float64
 9   Changed_Credit_Limit      100000 non-null  float64
 10  Num_Credit_Inquiries      98035 non-null   float64
 11  Outstanding_Debt          100000 non-null  float64
 12  Credit_Utilization_Ratio  100000 non-null  float64
 13  Credit_History_Age        90970 non-null   fl

In [14]:
numeric_df.describe()

,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Total_EMI_per_month,Amount_invested_monthly,Monthly_Balance,num_month
count,"100,000.00","100,000.00","84,998.00","100,000.00","100,000.00","100,000.00","100,000.00","100,000.00","92,998.00","100,000.00","98,035.00","100,000.00","100,000.00","90,970.00","100,000.00","95,521.00","97,132.00","100,000.00"
mean,110.65,"176,415.70","4,194.17",17.09,22.47,72.47,3.01,21.07,30.92,10.17,27.75,"1,426.22",32.29,18.43,"1,403.12",637.41,"-30,885,804,884,075,277,713,408.00","202,204.50"
std,686.24,"1,429,618.05","3,183.69",117.40,129.06,466.42,62.65,14.86,226.03,6.88,193.18,"1,155.13",5.12,8.31,"8,306.04","2,043.32","3,208,491,911,407,571,723,878,400.00",2.29
min,-500.00,"7,005.93",303.65,-1.00,0.00,1.00,-100.00,-5.00,-3.00,-6.49,0.00,0.23,20.00,0.08,0.00,0.00,"-333,333,333,333,333,314,856,026,112.00","202,201.00"
25%,24.00,"19,457.50","1,625.57",3.00,4.00,8.00,1.00,10.00,9.00,4.97,3.00,566.07,28.05,12.00,30.31,74.53,269.99,"202,202.75"
50%,33.00,"37,578.61","3,093.75",6.00,5.00,13.00,3.00,18.00,14.00,9.25,6.00,"1,166.15",32.31,18.25,69.25,135.93,336.45,"202,204.50"
75%,42.00,"72,790.92","5,957.45",7.00,7.00,20.00,5.00,28.00,18.00,14.66,9.00,"1,945.96",36.50,25.17,161.22,265.73,469.64,"202,206.25"
max,"8,698.00","24,198,062.00","15,204.63","1,798.00","1,499.00","5,797.00","1,496.00",67.00,"4,397.00",36.97,"2,597.00","4,998.07",50.00,33.67,"82,331.00","10,000.00","1,602.04","202,208.00"


# Cleaning Categorical values

In [15]:
categorical_variables = [
    "Occupation",
    "Type_of_Loan",
    "Credit_Mix",
    "Payment_Behaviour",
    "Payment_of_Min_Amount",
]

## Payment Behaviour

In [16]:
train_df["Payment_Behaviour"].value_counts()

Payment_Behaviour
Low_spent_Small_value_payments      25513
High_spent_Medium_value_payments    17540
Low_spent_Medium_value_payments     13861
High_spent_Large_value_payments     13721
High_spent_Small_value_payments     11340
Low_spent_Large_value_payments      10425
!@9#%8                               7600
Name: count, dtype: int64

In [17]:
train_df[train_df["Payment_Behaviour"] == "!@9#%8"].head()

,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score,num_month
5,0x1607,CUS_0xd40,June,Aaron Maashoh,23,821-00-0265,Scientist,"19,114.12",NaN,3,4,3,4,"Auto Loan, Credit-Builder Loan, Personal Loan,...",8,4.00,9.27,4.00,Good,809.98,27.26,22.50,No,49.57,62.43,!@9#%8,340.48,Good,202206
16,0x161a,CUS_0x2dbc,January,Langep,34,486-85-3974,_______,"143,162.64","12,187.22",1,5,8,3,"Auto Loan, Auto Loan, and Not Specified",5,8.00,7.10,3.00,Good,"1,303.01",28.62,17.75,No,246.99,168.41,!@9#%8,"1,043.32",Good,202201
32,0x1632,CUS_0x1cdb,January,Deepaa,21,615-06-7821,Developer,"35,547.71","2,853.31",7,5,5,0,NaN,5,NaN,2.58,4.00,Standard,943.86,39.80,30.67,Yes,0.00,276.73,!@9#%8,288.61,Standard,202201
47,0x1645,CUS_0x95ee,August,Np,31,612-70-8987,Lawyer,"73,928.46","5,988.71",4,5,8,0,NaN,8,7.00,10.14,NaN,Good,548.20,31.58,32.50,No,0.00,42.64,!@9#%8,796.23,Good,202208
54,0x1650,CUS_0x284a,July,Nadiaq,34,#F%$D@*&8,Lawyer,"10,909,427.00",NaN,0,1,8,2,"Credit-Builder Loan, and Mortgage Loan",0,2.00,9.34,4.00,Good,352.16,26.95,31.08,No,911.22,930.39,!@9#%8,326.24,Good,202207


In [18]:
train_df[train_df["Customer_ID"] == "CUS_0xd40"]

,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score,num_month
0,0x1602,CUS_0xd40,January,Aaron Maashoh,23,821-00-0265,Scientist,"19,114.12","1,824.84",3,4,3,4,"Auto Loan, Credit-Builder Loan, Personal Loan,...",3,7.00,11.27,4.00,_,809.98,26.82,22.08,No,49.57,80.42,High_spent_Small_value_payments,312.49,Good,202201
1,0x1603,CUS_0xd40,February,Aaron Maashoh,23,821-00-0265,Scientist,"19,114.12",NaN,3,4,3,4,"Auto Loan, Credit-Builder Loan, Personal Loan,...",-1,NaN,11.27,4.00,Good,809.98,31.94,NaN,No,49.57,118.28,Low_spent_Large_value_payments,284.63,Good,202202
2,0x1604,CUS_0xd40,March,Aaron Maashoh,-500,821-00-0265,Scientist,"19,114.12",NaN,3,4,3,4,"Auto Loan, Credit-Builder Loan, Personal Loan,...",3,7.00,0.00,4.00,Good,809.98,28.61,22.25,No,49.57,81.70,Low_spent_Medium_value_payments,331.21,Good,202203
3,0x1605,CUS_0xd40,April,Aaron Maashoh,23,821-00-0265,Scientist,"19,114.12",NaN,3,4,3,4,"Auto Loan, Credit-Builder Loan, Personal Loan,...",5,4.00,6.27,4.00,Good,809.98,31.38,22.33,No,49.57,199.46,Low_spent_Small_value_payments,223.45,Good,202204
4,0x1606,CUS_0xd40,May,Aaron Maashoh,23,821-00-0265,Scientist,"19,114.12","1,824.84",3,4,3,4,"Auto Loan, Credit-Builder Loan, Personal Loan,...",6,NaN,11.27,4.00,Good,809.98,24.80,22.42,No,49.57,41.42,High_spent_Medium_value_payments,341.49,Good,202205
5,0x1607,CUS_0xd40,June,Aaron Maashoh,23,821-00-0265,Scientist,"19,114.12",NaN,3,4,3,4,"Auto Loan, Credit-Builder Loan, Personal Loan,...",8,4.00,9.27,4.00,Good,809.98,27.26,22.50,No,49.57,62.43,!@9#%8,340.48,Good,202206
6,0x1608,CUS_0xd40,July,Aaron Maashoh,23,821-00-0265,Scientist,"19,114.12","1,824.84",3,4,3,4,"Auto Loan, Credit-Builder Loan, Personal Loan,...",3,8.00,11.27,4.00,Good,809.98,22.54,22.58,No,49.57,178.34,Low_spent_Small_value_payments,244.57,Good,202207
7,0x1609,CUS_0xd40,August,NaN,23,#F%$D@*&8,Scientist,"19,114.12","1,824.84",3,4,3,4,"Auto Loan, Credit-Builder Loan, Personal Loan,...",3,6.00,11.27,4.00,Good,809.98,23.93,NaN,No,49.57,24.79,High_spent_Medium_value_payments,358.12,Standard,202208


In [19]:
train_df["Payment_Behaviour"] = train_df["Payment_Behaviour"].replace("!@9#%8", pd.NA)

## Occupation

In [20]:
train_df[train_df["Occupation"] == "_______"].head()

,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score,num_month
8,0x160e,CUS_0x21b1,January,Rick Rothackerj,28,004-07-5839,_______,"34,847.84","3,037.99",2,4,6,1,Credit-Builder Loan,3,4.00,5.42,2.00,Good,605.03,24.46,26.58,No,18.82,104.29,Low_spent_Small_value_payments,470.69,Standard,202201
16,0x161a,CUS_0x2dbc,January,Langep,34,486-85-3974,_______,"143,162.64","12,187.22",1,5,8,3,"Auto Loan, Auto Loan, and Not Specified",5,8.00,7.10,3.00,Good,"1,303.01",28.62,17.75,No,246.99,168.41,<NA>,"1,043.32",Good,202201
18,0x161c,CUS_0x2dbc,March,Langep,34,486-85-3974,_______,"143,162.64",NaN,1,5,8,3,"Auto Loan, Auto Loan, and Not Specified",8,7.00,11.10,NaN,Good,"1,303.01",26.52,17.92,No,246.99,"10,000.00",High_spent_Small_value_payments,715.74,Good,202203
20,0x161e,CUS_0x2dbc,May,Langep,34,486-85-3974,_______,"143,162.64","12,187.22",1,5,8,3,"Auto Loan, Auto Loan, and Not Specified",10,5.00,7.10,3.00,Good,"1,303.01",31.38,18.08,No,246.99,430.95,Low_spent_Large_value_payments,810.78,Good,202205
29,0x162b,CUS_0xb891,June,Jasond,55,#F%$D@*&8,_______,"30,689.89","2,612.49",2,5,4,1,Not Specified,5,6.00,-3.01,4.00,_,632.46,27.45,17.67,No,16.42,84.95,High_spent_Small_value_payments,419.88,Standard,202206


In [21]:
train_df[train_df["Customer_ID"] == "CUS_0x9e67"].head(5)

,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score,num_month
93280,0x23892,CUS_0x9e67,January,Marias,24,366-87-4584,_______,"44,393.86","3,504.49",6,3,12,2,"Student Loan, and Payday Loan",11,9.00,10.74,NaN,_,"1,270.97",37.33,NaN,Yes,63.79,91.47,High_spent_Medium_value_payments,445.18,Standard,202201
93281,0x23893,CUS_0x9e67,February,NaN,24,366-87-4584,Media_Manager,"44,393.86",NaN,6,3,12,2,"Student Loan, and Payday Loan",15,11.00,10.74,3.00,Standard,"1,270.97",38.29,20.33,Yes,63.79,48.74,High_spent_Large_value_payments,477.91,Standard,202202
93282,0x23894,CUS_0x9e67,March,Marias,24,366-87-4584,_______,"44,393.86","3,504.49",6,3,12,2,"Student Loan, and Payday Loan",15,11.00,10.74,3.00,_,"1,270.97",29.12,20.42,Yes,63.79,73.10,High_spent_Medium_value_payments,463.55,Standard,202203
93283,0x23895,CUS_0x9e67,April,Marias,24,366-87-4584,_______,"44,393.86","3,504.49",6,3,12,2,"Student Loan, and Payday Loan",15,13.00,0.00,3.00,_,"1,270.97",27.48,NaN,Yes,63.79,218.85,Low_spent_Small_value_payments,357.81,Standard,202204
93284,0x23896,CUS_0x9e67,May,NaN,24,366-87-4584,_______,"44,393.86","3,504.49",6,3,12,2,"Student Loan, and Payday Loan",19,11.00,10.74,3.00,Standard,"1,270.97",25.37,20.58,Yes,63.79,129.96,Low_spent_Large_value_payments,426.69,Poor,202205


In [22]:
train_df[train_df["Occupation"] == "_______"].groupby("Month").size()

Month
April       928
August      880
February    900
January     827
July        823
June        948
March       877
May         879
dtype: int64

In [23]:
train_df.groupby("Customer_ID")["Occupation"].nunique().sort_values(ascending=False)

Customer_ID
CUS_0x6da9    2
CUS_0x4aef    2
CUS_0x74fc    2
CUS_0x74f7    2
CUS_0x74e8    2
             ..
CUS_0x7968    1
CUS_0x7967    1
CUS_0x367d    1
CUS_0x7960    1
CUS_0x1000    1
Name: Occupation, Length: 12500, dtype: int64

In [24]:
valid_occupations = (
    train_df[train_df["Occupation"] != "_______"]
    .groupby("Customer_ID")["Occupation"]
    .first()
)
train_df["Occupation_filled"] = train_df["Customer_ID"].map(valid_occupations)
train_df["Occupation_filled"]

0        Scientist
1        Scientist
2        Scientist
3        Scientist
4        Scientist
           ...    
99995     Mechanic
99996     Mechanic
99997     Mechanic
99998     Mechanic
99999     Mechanic
Name: Occupation_filled, Length: 100000, dtype: object

In [25]:
train_df.loc[train_df["Occupation"] == "_______", "Occupation"] = train_df[
    "Occupation_filled"
]
train_df["Occupation"].value_counts()

Occupation
Lawyer           7096
Engineer         6864
Architect        6824
Mechanic         6776
Scientist        6744
Accountant       6744
Developer        6720
Media_Manager    6720
Teacher          6672
Entrepreneur     6648
Doctor           6568
Journalist       6536
Manager          6432
Musician         6352
Writer           6304
Name: count, dtype: int64

## Type of loan

In [26]:
train_df["Type_of_Loan"] = train_df["Type_of_Loan"].apply(clean_type_of_loan)

In [27]:
unique_loans = set()

for loans in train_df["Type_of_Loan"]:
    unique_loans.update(loans)

unique_loans

{'Auto Loan',
 'Credit-Builder Loan',
 'Debt Consolidation Loan',
 'Home Equity Loan',
 'Mortgage Loan',
 'Not Specified',
 'Payday Loan',
 'Personal Loan',
 'Student Loan'}

In [28]:
for loan in unique_loans:
    train_df[loan] = train_df["Type_of_Loan"].apply(lambda x: int(loan in x))

In [29]:
unique_loans = list(unique_loans)
train_df[unique_loans]

,Payday Loan,Not Specified,Personal Loan,Mortgage Loan,Credit-Builder Loan,Home Equity Loan,Student Loan,Debt Consolidation Loan,Auto Loan
0,0,0,1,0,1,1,0,0,1
1,0,0,1,0,1,1,0,0,1
2,0,0,1,0,1,1,0,0,1
3,0,0,1,0,1,1,0,0,1
4,0,0,1,0,1,1,0,0,1
...,...,...,...,...,...,...,...,...,...
99995,0,0,0,0,0,0,1,0,1
99996,0,0,0,0,0,0,1,0,1
99997,0,0,0,0,0,0,1,0,1
99998,0,0,0,0,0,0,1,0,1


## Credit Mix

In [30]:
train_df["Credit_Mix"].value_counts(normalize=True)

Credit_Mix
Standard   0.36
Good       0.24
_          0.20
Bad        0.19
Name: proportion, dtype: float64

In [31]:
train_df[train_df["Credit_Mix"] == "_"].head()

,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score,num_month,Occupation_filled,Payday Loan,Not Specified,Personal Loan,Mortgage Loan,Credit-Builder Loan,Home Equity Loan,Student Loan,Debt Consolidation Loan,Auto Loan
0,0x1602,CUS_0xd40,January,Aaron Maashoh,23,821-00-0265,Scientist,"19,114.12","1,824.84",3,4,3,4,"[Auto Loan, Credit-Builder Loan, Personal Loan...",3,7.00,11.27,4.00,_,809.98,26.82,22.08,No,49.57,80.42,High_spent_Small_value_payments,312.49,Good,202201,Scientist,0,0,1,0,1,1,0,0,1
10,0x1610,CUS_0x21b1,March,Rick Rothackerj,28,004-07-5839,Teacher,"34,847.84","3,037.99",2,1385,6,1,[Credit-Builder Loan],3,-1.00,5.42,2.00,_,605.03,33.22,26.75,No,18.82,58.52,High_spent_Large_value_payments,466.47,Standard,202203,Teacher,0,0,0,0,1,0,0,0,0
19,0x161d,CUS_0x2dbc,April,Langep,34,486-85-3974,Engineer,"143,162.64","12,187.22",1,5,8,3,"[Auto Loan, Auto Loan, Not Specified]",8,5.00,9.10,3.00,_,"1,303.01",39.50,NaN,No,246.99,825.22,Low_spent_Medium_value_payments,426.51,Good,202204,Engineer,0,1,0,0,0,0,0,0,1
29,0x162b,CUS_0xb891,June,Jasond,55,#F%$D@*&8,Entrepreneur,"30,689.89","2,612.49",2,5,4,1,[Not Specified],5,6.00,-3.01,4.00,_,632.46,27.45,17.67,No,16.42,84.95,High_spent_Small_value_payments,419.88,Standard,202206,Entrepreneur,0,1,0,0,0,0,0,0,0
35,0x1635,CUS_0x1cdb,April,Deepaa,21,615-06-7821,Developer,"35,547.71","2,853.31",7,5,5,0,[],1,15.00,2.58,4.00,_,943.86,28.92,30.92,Yes,0.00,96.79,High_spent_Medium_value_payments,438.55,Standard,202204,Developer,0,0,0,0,0,0,0,0,0


In [32]:
train_df[train_df["Customer_ID"] == "CUS_0x1cdb"].head()

,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score,num_month,Occupation_filled,Payday Loan,Not Specified,Personal Loan,Mortgage Loan,Credit-Builder Loan,Home Equity Loan,Student Loan,Debt Consolidation Loan,Auto Loan
32,0x1632,CUS_0x1cdb,January,Deepaa,21,615-06-7821,Developer,"35,547.71","2,853.31",7,5,5,0,[],5,NaN,2.58,4.00,Standard,943.86,39.80,30.67,Yes,0.00,276.73,<NA>,288.61,Standard,202201,Developer,0,0,0,0,0,0,0,0,0
33,0x1633,CUS_0x1cdb,February,Deepaa,21,615-06-7821,Developer,"35,547.71",NaN,7,5,5,0,[],9,NaN,2.58,4.00,Standard,943.86,27.02,30.75,NM,0.00,74.44,High_spent_Medium_value_payments,460.89,Standard,202202,Developer,0,0,0,0,0,0,0,0,0
34,0x1634,CUS_0x1cdb,March,Deepaa,21,615-06-7821,Developer,"35,547.71","2,853.31",7,5,5,-100,[],5,12.00,2.58,4.00,Standard,943.86,23.46,30.83,Yes,0.00,173.14,Low_spent_Medium_value_payments,392.19,Standard,202203,Developer,0,0,0,0,0,0,0,0,0
35,0x1635,CUS_0x1cdb,April,Deepaa,21,615-06-7821,Developer,"35,547.71","2,853.31",7,5,5,0,[],1,15.00,2.58,4.00,_,943.86,28.92,30.92,Yes,0.00,96.79,High_spent_Medium_value_payments,438.55,Standard,202204,Developer,0,0,0,0,0,0,0,0,0
36,0x1636,CUS_0x1cdb,May,Deepaa,21,615-06-7821,Developer,"35,547.71","2,853.31",7,5,5,0,[],9,17.00,2.58,4.00,_,943.86,41.78,31.00,Yes,0.00,62.72,High_spent_Small_value_payments,482.61,Standard,202205,Developer,0,0,0,0,0,0,0,0,0


In [33]:
train_df.groupby("Customer_ID")["Credit_Mix"].nunique().sort_values(ascending=False)

Customer_ID
CUS_0x6da9    2
CUS_0x8804    2
CUS_0x87c2    2
CUS_0x87c9    2
CUS_0x87d0    2
             ..
CUS_0x2db5    1
CUS_0x7d4b    1
CUS_0x2db3    1
CUS_0x2da9    1
CUS_0x1000    1
Name: Credit_Mix, Length: 12500, dtype: int64

In [34]:
df_test_cm = train_df[train_df["Credit_Mix"] != "_"]
df_test_cm.groupby("Customer_ID")["Credit_Mix"].nunique().sort_values(ascending=False)

Customer_ID
CUS_0x1000    1
CUS_0x8cfe    1
CUS_0x8cbe    1
CUS_0x8cc1    1
CUS_0x8cc5    1
             ..
CUS_0x4f3b    1
CUS_0x4f3e    1
CUS_0x4f41    1
CUS_0x4f43    1
CUS_0xffd     1
Name: Credit_Mix, Length: 12500, dtype: int64

In [35]:
df_test_cm = (
    train_df[train_df["Credit_Mix"] != "_"].groupby("Customer_ID")["Credit_Mix"].first()
)

train_df["Credit_Mix_Filled"] = train_df["Customer_ID"].map(df_test_cm)
train_df["Credit_Mix_Filled"]

0        Good
1        Good
2        Good
3        Good
4        Good
         ... 
99995    Good
99996    Good
99997    Good
99998    Good
99999    Good
Name: Credit_Mix_Filled, Length: 100000, dtype: object

In [37]:
train_df.loc[train_df["Credit_Mix"] == "_", "Credit_Mix"] = train_df[
    "Credit_Mix_Filled"
]
train_df["Credit_Mix"].value_counts()

Credit_Mix
Standard    45848
Good        30384
Bad         23768
Name: count, dtype: int64

## Payment of min amount

In [38]:
train_df[train_df["Payment_of_Min_Amount"] == "NM"].head()

,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score,num_month,Occupation_filled,Payday Loan,Not Specified,Personal Loan,Mortgage Loan,Credit-Builder Loan,Home Equity Loan,Student Loan,Debt Consolidation Loan,Auto Loan,Credit_Mix_Filled
14,0x1614,CUS_0x21b1,July,Rick Rothackerj,28,004-07-5839,Teacher,"34,847.84",NaN,2,4,6,1,[Credit-Builder Loan],3,4.00,5.42,2.00,Good,605.03,31.13,27.08,NM,18.82,70.10,High_spent_Medium_value_payments,464.88,Good,202207,Teacher,0,0,0,0,1,0,0,0,0,Good
26,0x1628,CUS_0xb891,March,Jasond,55,072-31-6145,Entrepreneur,"30,689.89","2,612.49",2,5,4,1,[Not Specified],3,9.00,1.99,4.00,Good,632.46,32.30,17.42,NM,16.42,83.41,High_spent_Medium_value_payments,411.43,Standard,202203,Entrepreneur,0,1,0,0,0,0,0,0,0,Good
33,0x1633,CUS_0x1cdb,February,Deepaa,21,615-06-7821,Developer,"35,547.71",NaN,7,5,5,0,[],9,NaN,2.58,4.00,Standard,943.86,27.02,30.75,NM,0.00,74.44,High_spent_Medium_value_payments,460.89,Standard,202202,Developer,0,0,0,0,0,0,0,0,0,Standard
41,0x163f,CUS_0x95ee,February,Np,31,612-70-8987,Lawyer,"73,928.46","5,988.71",4,5,8,0,[],8,7.00,10.14,2.00,Good,548.20,42.77,32.00,NM,0.00,172.94,Low_spent_Medium_value_payments,705.93,Good,202202,Lawyer,0,0,0,0,0,0,0,0,0,Good
48,0x164a,CUS_0x284a,January,Nadiaq,33,411-51-0676,Lawyer,"131,313.40","11,242.78",0,1,8,2,"[Credit-Builder Loan, Mortgage Loan]",0,3.00,9.34,2.00,Good,352.16,32.20,30.58,NM,137.64,378.17,High_spent_Medium_value_payments,858.46,Good,202201,Lawyer,0,0,0,1,1,0,0,0,0,Good


In [39]:
train_df[train_df["Customer_ID"] == "CUS_0x21b1"].head()

,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score,num_month,Occupation_filled,Payday Loan,Not Specified,Personal Loan,Mortgage Loan,Credit-Builder Loan,Home Equity Loan,Student Loan,Debt Consolidation Loan,Auto Loan,Credit_Mix_Filled
8,0x160e,CUS_0x21b1,January,Rick Rothackerj,28,004-07-5839,Teacher,"34,847.84","3,037.99",2,4,6,1,[Credit-Builder Loan],3,4.00,5.42,2.00,Good,605.03,24.46,26.58,No,18.82,104.29,Low_spent_Small_value_payments,470.69,Standard,202201,Teacher,0,0,0,0,1,0,0,0,0,Good
9,0x160f,CUS_0x21b1,February,Rick Rothackerj,28,004-07-5839,Teacher,"34,847.84","3,037.99",2,4,6,1,[Credit-Builder Loan],7,1.00,7.42,2.00,Good,605.03,38.55,26.67,No,18.82,40.39,High_spent_Large_value_payments,484.59,Good,202202,Teacher,0,0,0,0,1,0,0,0,0,Good
10,0x1610,CUS_0x21b1,March,Rick Rothackerj,28,004-07-5839,Teacher,"34,847.84","3,037.99",2,1385,6,1,[Credit-Builder Loan],3,-1.00,5.42,2.00,Good,605.03,33.22,26.75,No,18.82,58.52,High_spent_Large_value_payments,466.47,Standard,202203,Teacher,0,0,0,0,1,0,0,0,0,Good
11,0x1611,CUS_0x21b1,April,Rick Rothackerj,28,004-07-5839,Teacher,"34,847.84",NaN,2,4,6,1,[Credit-Builder Loan],3,3.00,5.42,2.00,Good,605.03,39.18,26.83,No,18.82,99.31,Low_spent_Medium_value_payments,465.68,Good,202204,Teacher,0,0,0,0,1,0,0,0,0,Good
12,0x1612,CUS_0x21b1,May,Rick Rothackerj,28,004-07-5839,Teacher,"34,847.84","3,037.99",2,4,6,1,[Credit-Builder Loan],3,1.00,6.42,2.00,Good,605.03,34.98,26.92,No,18.82,130.12,Low_spent_Small_value_payments,444.87,Good,202205,Teacher,0,0,0,0,1,0,0,0,0,Good


In [40]:
df_test_cm = train_df[train_df["Payment_of_Min_Amount"] != "NM"]
df_test_cm.groupby("Customer_ID")["Payment_of_Min_Amount"].nunique().sort_values(
    ascending=False
)

Customer_ID
CUS_0x1000    1
CUS_0x8cfe    1
CUS_0x8cbe    1
CUS_0x8cc1    1
CUS_0x8cc5    1
             ..
CUS_0x4f3b    1
CUS_0x4f3e    1
CUS_0x4f41    1
CUS_0x4f43    1
CUS_0xffd     1
Name: Payment_of_Min_Amount, Length: 12500, dtype: int64

In [41]:
df_group_pomu = (
    train_df[train_df["Payment_of_Min_Amount"] != "NM"]
    .groupby("Customer_ID")["Payment_of_Min_Amount"]
    .first()
)

train_df["Payment_of_Min_Amount"] = train_df["Customer_ID"].map(df_group_pomu)
train_df["Payment_of_Min_Amount"]

0        No
1        No
2        No
3        No
4        No
         ..
99995    No
99996    No
99997    No
99998    No
99999    No
Name: Payment_of_Min_Amount, Length: 100000, dtype: object

In [42]:
train_df.loc[train_df["Payment_of_Min_Amount"] == "NM", "Payment_of_Min_Amount"] = (
    train_df["Credit_Mix_Filled"]
)
train_df["Payment_of_Min_Amount"].value_counts()

Payment_of_Min_Amount
Yes    59432
No     40568
Name: count, dtype: int64

# Null values

In [43]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 40 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   ID                        100000 non-null  object 
 1   Customer_ID               100000 non-null  object 
 2   Month                     100000 non-null  object 
 3   Name                      90015 non-null   object 
 4   Age                       100000 non-null  int64  
 5   SSN                       100000 non-null  object 
 6   Occupation                100000 non-null  object 
 7   Annual_Income             100000 non-null  float64
 8   Monthly_Inhand_Salary     84998 non-null   float64
 9   Num_Bank_Accounts         100000 non-null  int64  
 10  Num_Credit_Card           100000 non-null  int64  
 11  Interest_Rate             100000 non-null  int64  
 12  Num_of_Loan               100000 non-null  int64  
 13  Type_of_Loan              100000 non-null  ob

In [44]:
numeric_variables = [
    "Age",
    "Annual_Income",
    "Monthly_Inhand_Salary",
    "Num_Bank_Accounts",
    "Interest_Rate",
    "Num_of_Loan",
    "Delay_from_due_date",
    "Changed_Credit_Limit",
    "Num_Credit_Inquiries",
    "Outstanding_Debt",
    "Credit_Utilization_Ratio",
    "Credit_History_Age",
    "Total_EMI_per_month",
    "Amount_invested_monthly",
    "Monthly_Balance",
    "Num_of_Delayed_Payment",
]

categorical_variables = [
    "Occupation",
    "Credit_Mix",
    "Payment_Behaviour",
    "Payment_of_Min_Amount",
] + list(unique_loans)


ids = ["Customer_ID", "num_month"]

final_features = {
    "ids": ids,
    "numeric": numeric_variables,
    "categorical": categorical_variables,
}

In [45]:
final_features["categorical"]

['Occupation',
 'Credit_Mix',
 'Payment_Behaviour',
 'Payment_of_Min_Amount',
 'Payday Loan',
 'Not Specified',
 'Personal Loan',
 'Mortgage Loan',
 'Credit-Builder Loan',
 'Home Equity Loan',
 'Student Loan',
 'Debt Consolidation Loan',
 'Auto Loan']

In [46]:
df_train_clean = train_df[
    final_features["ids"] + final_features["numeric"] + final_features["categorical"]
]

In [47]:
missing = pd.DataFrame(
    {
        "missing_count": df_train_clean.isna().sum(),
        "missing_pct": df_train_clean.isna().mean() * 100,
    }
)

missing = missing.sort_values("missing_pct", ascending=False)

missing.head(10)

,missing_count,missing_pct
Monthly_Inhand_Salary,15002,15.00
Credit_History_Age,9030,9.03
Payment_Behaviour,7600,7.60
Num_of_Delayed_Payment,7002,7.00
Amount_invested_monthly,4479,4.48
Monthly_Balance,2868,2.87
Num_Credit_Inquiries,1965,1.97
Home Equity Loan,0,0.00
Credit-Builder Loan,0,0.00
Mortgage Loan,0,0.00


## Monthly Inhand Salary

In [48]:
df_train_clean[df_train_clean["Monthly_Inhand_Salary"].isna()]

,Customer_ID,num_month,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Interest_Rate,Num_of_Loan,Delay_from_due_date,Changed_Credit_Limit,Num_Credit_Inquiries,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Total_EMI_per_month,Amount_invested_monthly,Monthly_Balance,Num_of_Delayed_Payment,Occupation,Credit_Mix,Payment_Behaviour,Payment_of_Min_Amount,Payday Loan,Not Specified,Personal Loan,Mortgage Loan,Credit-Builder Loan,Home Equity Loan,Student Loan,Debt Consolidation Loan,Auto Loan
1,CUS_0xd40,202202,23,"19,114.12",NaN,3,3,4,-1,11.27,4.00,809.98,31.94,NaN,49.57,118.28,284.63,NaN,Scientist,Good,Low_spent_Large_value_payments,No,0,0,1,0,1,1,0,0,1
2,CUS_0xd40,202203,-500,"19,114.12",NaN,3,3,4,3,0.00,4.00,809.98,28.61,22.25,49.57,81.70,331.21,7.00,Scientist,Good,Low_spent_Medium_value_payments,No,0,0,1,0,1,1,0,0,1
3,CUS_0xd40,202204,23,"19,114.12",NaN,3,3,4,5,6.27,4.00,809.98,31.38,22.33,49.57,199.46,223.45,4.00,Scientist,Good,Low_spent_Small_value_payments,No,0,0,1,0,1,1,0,0,1
5,CUS_0xd40,202206,23,"19,114.12",NaN,3,3,4,8,9.27,4.00,809.98,27.26,22.50,49.57,62.43,340.48,4.00,Scientist,Good,<NA>,No,0,0,1,0,1,1,0,0,1
11,CUS_0x21b1,202204,28,"34,847.84",NaN,2,6,1,3,5.42,2.00,605.03,39.18,26.83,18.82,99.31,465.68,3.00,Teacher,Good,Low_spent_Medium_value_payments,No,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99944,CUS_0x51b3,202201,33,"59,146.36",NaN,2,6,1,8,6.68,2.00,418.03,34.72,NaN,26.78,606.44,NaN,6.00,Media_Manager,Good,Low_spent_Small_value_payments,No,0,0,1,0,0,0,0,0,0
99955,CUS_0x2084,202204,21,"38,321.39",NaN,4,3,4,11,1.59,3.00,678.57,35.46,17.17,362.07,293.71,NaN,7.00,Architect,Good,Low_spent_Large_value_payments,No,0,0,0,1,0,0,1,1,0
99963,CUS_0x372c,202204,-500,"42,903.79",NaN,0,6,1,14,5.10,1.00,"1,079.48",30.63,NaN,34.98,31.19,NaN,0.00,Lawyer,Good,High_spent_Large_value_payments,No,0,1,0,0,0,0,0,0,0
99975,CUS_0xf16,202208,45,"16,680.35",NaN,1,5,4,1,5.69,8.00,897.16,41.21,NaN,41.11,70.81,NaN,0.00,Media_Manager,Good,Low_spent_Large_value_payments,No,1,1,0,1,0,0,1,0,0


In [49]:
df_test_mhs = df_train_clean[df_train_clean["Monthly_Inhand_Salary"].notna()]
df_test_mhs.groupby("Customer_ID")["Monthly_Inhand_Salary"].nunique().sort_values(
    ascending=False
).head(10)

Customer_ID
CUS_0xbc9c    2
CUS_0xbba8    2
CUS_0xa503    2
CUS_0x4e5     2
CUS_0xa514    2
CUS_0x634d    2
CUS_0x4e71    2
CUS_0x9093    2
CUS_0x2084    2
CUS_0xa54d    2
Name: Monthly_Inhand_Salary, dtype: int64

In [50]:
df_train_clean[df_train_clean["Customer_ID"] == "CUS_0x634d"]

,Customer_ID,num_month,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Interest_Rate,Num_of_Loan,Delay_from_due_date,Changed_Credit_Limit,Num_Credit_Inquiries,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Total_EMI_per_month,Amount_invested_monthly,Monthly_Balance,Num_of_Delayed_Payment,Occupation,Credit_Mix,Payment_Behaviour,Payment_of_Min_Amount,Payday Loan,Not Specified,Personal Loan,Mortgage Loan,Credit-Builder Loan,Home Equity Loan,Student Loan,Debt Consolidation Loan,Auto Loan
38800,CUS_0x634d,202201,40,"17,594.01","1,657.17",0,11,0,2,1.52,0.00,576.75,40.00,27.92,0.00,43.07,382.64,8.00,Engineer,Good,<NA>,No,0,0,0,0,0,0,0,0,0
38801,CUS_0x634d,202202,40,"17,594.01","1,657.17",0,11,0,2,1.52,0.00,576.75,34.97,28.00,"9,912.00",24.75,390.96,NaN,Engineer,Good,High_spent_Medium_value_payments,No,0,0,0,0,0,0,0,0,0
38802,CUS_0x634d,202203,40,"17,594.01","1,657.17",0,11,0,2,1.52,0.00,576.75,25.41,28.08,0.00,NaN,285.07,10.00,Engineer,Good,Low_spent_Medium_value_payments,No,0,0,0,0,0,0,0,0,0
38803,CUS_0x634d,202204,40,"17,594.01","1,657.17",0,11,0,2,1.52,0.00,576.75,24.97,28.17,0.00,89.99,365.73,13.00,Engineer,Good,Low_spent_Small_value_payments,No,0,0,0,0,0,0,0,0,0
38804,CUS_0x634d,202205,40,"17,594.01",NaN,0,11,0,2,-5.48,0.00,576.75,29.83,28.25,0.00,95.50,330.22,11.00,Engineer,Good,High_spent_Small_value_payments,No,0,0,0,0,0,0,0,0,0
38805,CUS_0x634d,202206,40,"17,594.01","1,657.17",0,3312,0,3,1.52,0.00,576.75,29.42,28.33,0.00,23.13,382.59,10.00,Engineer,Good,High_spent_Large_value_payments,No,0,0,0,0,0,0,0,0,0
38806,CUS_0x634d,202207,40,"17,594.01","1,485.44",0,11,0,3,1.52,0.00,576.75,37.15,28.42,171.73,27.51,388.21,10.00,Engineer,Good,High_spent_Medium_value_payments,No,0,0,0,0,0,0,0,0,0
38807,CUS_0x634d,202208,40,"17,594.01","1,485.44",0,11,0,2,8.52,5.00,576.75,27.17,28.50,171.73,34.15,381.56,10.00,Engineer,Good,High_spent_Medium_value_payments,No,0,0,0,0,0,0,0,0,0


In [51]:
df_train_clean["mhs_missing"] = (
    df_train_clean["Monthly_Inhand_Salary"].isna().astype(int)
)

/var/folders/1f/54wv342x795c_zyzqlgyjjkw0000gn/T/ipykernel_1573/2283952083.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train_clean["mhs_missing"] = (


TODO: Add this to the summarize
We can conclude that the null values in the feature monthly in hand salary do not correlate strongly with any other feature (in a basic analysis)

In [52]:
df_train_clean.groupby("mhs_missing")[final_features["numeric"]].mean()

,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Interest_Rate,Num_of_Loan,Delay_from_due_date,Changed_Credit_Limit,Num_Credit_Inquiries,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Total_EMI_per_month,Amount_invested_monthly,Monthly_Balance,Num_of_Delayed_Payment
mhs_missing,,,,,,,,,,,,,,,,
0,110.90,"176,799.61","4,194.17",16.89,72.95,3.21,21.10,10.19,27.84,"1,427.16",32.29,18.42,"1,404.62",637.49,"-32,315,789,898,891,971,788,800.00",31.07
1,109.24,"174,240.55",NaN,18.26,69.75,1.88,20.87,10.07,27.25,"1,420.90",32.26,18.48,"1,394.63",636.99,"-22,810,739,296,060,582,133,760.00",30.09


In [53]:
df_train_clean.groupby("mhs_missing")["Auto Loan"].value_counts(normalize=True)

mhs_missing  Auto Loan
0            0           0.70
             1           0.30
1            0           0.69
             1           0.31
Name: proportion, dtype: float64

In [54]:
# here we are filling the N/A values with the closer value , if is not the first the consecutive salary of the same costumer
df_train_clean = df_train_clean.sort_values(by=["Customer_ID", "num_month"])
df_train_clean["Monthly_Inhand_Salary"] = df_train_clean.groupby("Customer_ID")[
    "Monthly_Inhand_Salary"
].transform(lambda x: x.ffill().bfill())

## Credit_History_Age

In [56]:
df_train_clean[df_train_clean["Credit_History_Age"].isna()]

,Customer_ID,num_month,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Interest_Rate,Num_of_Loan,Delay_from_due_date,Changed_Credit_Limit,Num_Credit_Inquiries,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Total_EMI_per_month,Amount_invested_monthly,Monthly_Balance,Num_of_Delayed_Payment,Occupation,Credit_Mix,Payment_Behaviour,Payment_of_Min_Amount,Payday Loan,Not Specified,Personal Loan,Mortgage Loan,Credit-Builder Loan,Home Equity Loan,Student Loan,Debt Consolidation Loan,Auto Loan,mhs_missing
13763,CUS_0x1009,202204,26,"52,312.68","4,250.39",6,17,1094,10,9.73,2.00,202.68,25.00,NaN,108.37,171.45,405.22,18.00,Mechanic,Standard,High_spent_Small_value_payments,Yes,1,1,0,0,1,1,0,0,0,0
1529,CUS_0x100b,202202,18,"113,781.39","9,549.78",1,1,0,14,11.34,"2,271.00","1,030.20",35.95,NaN,0.00,661.62,563.35,9.00,Media_Manager,Good,Low_spent_Large_value_payments,No,0,0,0,0,0,0,0,0,0,0
95220,CUS_0x1013,202205,44,"98,620.98","7,962.42",3,6,3,12,1.33,3.00,"1,233.51",35.65,NaN,228.02,159.13,659.10,9.00,Mechanic,Good,High_spent_Medium_value_payments,No,0,0,1,0,0,0,1,1,0,0
65209,CUS_0x1015,202202,27,"46,951.02","3,725.59",7,16,0,8,15.83,4.00,340.22,25.57,NaN,0.00,114.33,538.23,9.00,Journalist,Standard,Low_spent_Medium_value_payments,Yes,0,0,0,0,0,0,0,0,0,0
82066,CUS_0x1018,202203,15,"61,194.81","5,014.57",7,23,8,24,28.63,8.00,"2,773.09",36.45,NaN,225.37,420.13,135.96,22.00,Accountant,Bad,<NA>,Yes,1,1,1,0,1,1,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63180,CUS_0xfea,202205,33,"44,264.41","3,755.70",3,6,2,15,9.66,3.00,"1,312.23",31.96,NaN,40.48,136.33,448.76,2.00,Manager,Good,High_spent_Medium_value_payments,No,0,0,0,0,0,0,1,0,0,0
5175,CUS_0xff3,202208,55,"17,032.78","1,176.40",0,2,3,13,6.86,5.00,"1,229.08",26.92,NaN,33.30,81.20,293.14,7.00,Scientist,Good,Low_spent_Small_value_payments,No,0,0,1,1,0,0,0,0,1,0
117,CUS_0xff4,202206,37,"25,546.26","2,415.86",8,14,5,16,7.83,5.00,758.44,34.59,NaN,101.33,61.73,338.53,10.00,Entrepreneur,Standard,High_spent_Small_value_payments,Yes,0,1,0,0,1,0,1,0,1,0
12953,CUS_0xffc,202202,17,"60,877.17","5,218.10",6,27,8,46,8.82,13.00,"1,300.13",34.64,NaN,272.81,105.80,403.21,16.00,Musician,Bad,High_spent_Small_value_payments,Yes,1,1,0,0,1,1,1,0,0,0


In [61]:
df_train_clean[df_train_clean["Customer_ID"] == "CUS_0xffc"]

,Customer_ID,num_month,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Interest_Rate,Num_of_Loan,Delay_from_due_date,Changed_Credit_Limit,Num_Credit_Inquiries,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Total_EMI_per_month,Amount_invested_monthly,Monthly_Balance,Num_of_Delayed_Payment,Occupation,Credit_Mix,Payment_Behaviour,Payment_of_Min_Amount,Payday Loan,Not Specified,Personal Loan,Mortgage Loan,Credit-Builder Loan,Home Equity Loan,Student Loan,Debt Consolidation Loan,Auto Loan,mhs_missing
12952,CUS_0xffc,202201,17,"60,877.17","5,218.10",6,27,8,46,5.82,8.00,"1,300.13",36.01,12.58,272.81,305.08,223.92,16.00,Musician,Bad,Low_spent_Medium_value_payments,Yes,1,1,0,0,1,1,1,0,0,0
12953,CUS_0xffc,202202,17,"60,877.17","5,218.10",6,27,8,46,8.82,13.00,"1,300.13",34.64,NaN,272.81,105.80,403.21,16.00,Musician,Bad,High_spent_Small_value_payments,Yes,1,1,0,0,1,1,1,0,0,0
12954,CUS_0xffc,202203,17,"60,877.17","5,218.10",6,27,8,46,7.82,13.00,"1,300.13",38.29,12.75,272.81,170.97,328.03,16.00,Musician,Bad,High_spent_Medium_value_payments,Yes,1,1,0,0,1,1,1,0,0,0
12955,CUS_0xffc,202204,17,"60,877.17","5,218.10",6,27,8,46,12.82,13.00,"1,300.13",37.61,12.83,272.81,185.35,333.65,14.00,Musician,Bad,Low_spent_Large_value_payments,Yes,1,1,0,0,1,1,1,0,0,0
12956,CUS_0xffc,202205,17,"60,877.17","5,218.10",6,27,8,41,0.00,13.00,"1,300.13",35.52,12.92,272.81,126.67,362.34,16.00,Musician,Bad,High_spent_Large_value_payments,Yes,1,1,0,0,1,1,1,0,0,0
12957,CUS_0xffc,202206,18,"60,877.17","5,218.10",6,27,8,46,8.82,13.00,"1,300.13",37.79,13.00,272.81,"10,000.00",279.82,19.00,Musician,Bad,Low_spent_Small_value_payments,Yes,1,1,0,0,1,1,1,0,0,0
12958,CUS_0xffc,202207,18,"60,877.17","5,218.10",6,27,8,46,8.82,13.00,"1,300.13",28.90,13.08,272.81,152.92,346.08,19.00,Musician,Bad,High_spent_Medium_value_payments,Yes,1,1,0,0,1,1,1,0,0,0
12959,CUS_0xffc,202208,18,"60,877.17","5,218.10",6,27,8,46,8.82,13.00,"1,300.13",29.03,NaN,272.81,46.43,442.57,14.00,Musician,Bad,High_spent_Large_value_payments,Yes,1,1,0,0,1,1,1,0,0,0


In [51]:
df_train_clean = df_train_clean.sort_values(["Customer_ID", "num_month"])

df_train_clean["Credit_History_Age"] = (
    df_train_clean.groupby("Customer_ID")["Credit_History_Age"]
    .apply(fill_credit_age)
    .reset_index(level=0, drop=True)
)

In [52]:
missing = pd.DataFrame(
    {
        "missing_count": df_train_clean.isna().sum(),
        "missing_pct": df_train_clean.isna().mean() * 100,
    }
)

missing = missing.sort_values("missing_pct", ascending=False)

missing.head(10)

,missing_count,missing_pct
Payment_Behaviour,7600,7.60
Num_of_Delayed_Payment,7002,7.00
Amount_invested_monthly,4479,4.48
Monthly_Balance,2868,2.87
Num_Credit_Inquiries,1965,1.97
Auto Loan,0,0.00
Credit_Mix,0,0.00
Payment_of_Min_Amount,0,0.00
Mortgage Loan,0,0.00
Student Loan,0,0.00


In [53]:
df_train_clean[df_train_clean["Customer_ID"] == "CUS_0xffc"]

,Customer_ID,num_month,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Interest_Rate,Num_of_Loan,Delay_from_due_date,Changed_Credit_Limit,Num_Credit_Inquiries,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Total_EMI_per_month,Amount_invested_monthly,Monthly_Balance,Num_of_Delayed_Payment,Occupation,Credit_Mix,Payment_Behaviour,Payment_of_Min_Amount,Mortgage Loan,Auto Loan,Student Loan,Not Specified,Home Equity Loan,Payday Loan,Personal Loan,Debt Consolidation Loan,Credit-Builder Loan,mhs_missing
12952,CUS_0xffc,202201,17,"60,877.17","5,218.10",6,27,8,46,5.82,8.00,"1,300.13",36.01,12.58,272.81,305.08,223.92,16.00,Musician,Bad,Low_spent_Medium_value_payments,Yes,0,0,1,1,1,1,0,0,1,0
12953,CUS_0xffc,202202,17,"60,877.17","5,218.10",6,27,8,46,8.82,13.00,"1,300.13",34.64,12.67,272.81,105.80,403.21,16.00,Musician,Bad,High_spent_Small_value_payments,Yes,0,0,1,1,1,1,0,0,1,0
12954,CUS_0xffc,202203,17,"60,877.17","5,218.10",6,27,8,46,7.82,13.00,"1,300.13",38.29,12.75,272.81,170.97,328.03,16.00,Musician,Bad,High_spent_Medium_value_payments,Yes,0,0,1,1,1,1,0,0,1,0
12955,CUS_0xffc,202204,17,"60,877.17","5,218.10",6,27,8,46,12.82,13.00,"1,300.13",37.61,12.83,272.81,185.35,333.65,14.00,Musician,Bad,Low_spent_Large_value_payments,Yes,0,0,1,1,1,1,0,0,1,0
12956,CUS_0xffc,202205,17,"60,877.17","5,218.10",6,27,8,41,0.00,13.00,"1,300.13",35.52,12.92,272.81,126.67,362.34,16.00,Musician,Bad,High_spent_Large_value_payments,Yes,0,0,1,1,1,1,0,0,1,0
12957,CUS_0xffc,202206,18,"60,877.17","5,218.10",6,27,8,46,8.82,13.00,"1,300.13",37.79,13.00,272.81,"10,000.00",279.82,19.00,Musician,Bad,Low_spent_Small_value_payments,Yes,0,0,1,1,1,1,0,0,1,0
12958,CUS_0xffc,202207,18,"60,877.17","5,218.10",6,27,8,46,8.82,13.00,"1,300.13",28.90,13.08,272.81,152.92,346.08,19.00,Musician,Bad,High_spent_Medium_value_payments,Yes,0,0,1,1,1,1,0,0,1,0
12959,CUS_0xffc,202208,18,"60,877.17","5,218.10",6,27,8,46,8.82,13.00,"1,300.13",29.03,13.17,272.81,46.43,442.57,14.00,Musician,Bad,High_spent_Large_value_payments,Yes,0,0,1,1,1,1,0,0,1,0


## Payment Behaviour

In [59]:
df_train_clean["Payment_Behaviour"].value_counts()

Payment_Behaviour
Low_spent_Small_value_payments      25513
High_spent_Medium_value_payments    17540
Low_spent_Medium_value_payments     13861
High_spent_Large_value_payments     13721
High_spent_Small_value_payments     11340
Low_spent_Large_value_payments      10425
Name: count, dtype: int64

In [57]:
df_train_clean[df_train_clean["Payment_Behaviour"].isna()].head()

,Customer_ID,num_month,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Interest_Rate,Num_of_Loan,Delay_from_due_date,Changed_Credit_Limit,Num_Credit_Inquiries,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Total_EMI_per_month,Amount_invested_monthly,Monthly_Balance,Num_of_Delayed_Payment,Occupation,Credit_Mix,Payment_Behaviour,Payment_of_Min_Amount,Mortgage Loan,Auto Loan,Student Loan,Not Specified,Home Equity Loan,Payday Loan,Personal Loan,Debt Consolidation Loan,Credit-Builder Loan,mhs_missing
56755,CUS_0x1000,202204,17,"30,625.94","2,706.16",6,27,2,64,1.63,11.00,"1,562.91",32.84,10.42,42.94,87.91,419.77,25.00,Lawyer,Bad,<NA>,Yes,0,0,0,0,1,0,0,0,1,0
13765,CUS_0x1009,202206,26,"52,312.68","4,250.39",6,17,4,3,9.73,4.00,202.68,30.61,30.83,108.37,150.80,445.88,18.00,Mechanic,Standard,<NA>,Yes,0,0,0,1,1,1,0,0,1,0
95218,CUS_0x1013,202203,44,"98,620.98","7,962.42",3,6,3,16,1.33,3.00,"1,233.51",29.66,17.42,228.02,323.43,524.80,9.00,Mechanic,Good,<NA>,No,0,0,1,0,0,0,1,1,0,0
65214,CUS_0x1015,202207,27,"46,951.02","3,725.59",7,16,0,8,15.83,9.00,340.22,38.45,21.33,0.00,459.55,203.01,9.00,Journalist,Standard,<NA>,Yes,0,0,0,0,0,0,0,0,0,0
82066,CUS_0x1018,202203,15,"61,194.81","5,014.57",7,23,8,24,28.63,8.00,"2,773.09",36.45,13.83,225.37,420.13,135.96,22.00,Accountant,Bad,<NA>,Yes,0,0,1,1,1,1,1,0,1,0


In [58]:
df_train_clean[df_train_clean["Customer_ID"] == "CUS_0x1000"]

,Customer_ID,num_month,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Interest_Rate,Num_of_Loan,Delay_from_due_date,Changed_Credit_Limit,Num_Credit_Inquiries,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Total_EMI_per_month,Amount_invested_monthly,Monthly_Balance,Num_of_Delayed_Payment,Occupation,Credit_Mix,Payment_Behaviour,Payment_of_Min_Amount,Mortgage Loan,Auto Loan,Student Loan,Not Specified,Home Equity Loan,Payday Loan,Personal Loan,Debt Consolidation Loan,Credit-Builder Loan,mhs_missing
56752,CUS_0x1000,202201,17,"30,625.94","2,706.16",6,27,2,62,1.63,10.00,"1,562.91",26.61,10.17,42.94,244.75,252.92,25.00,Lawyer,Bad,Low_spent_Large_value_payments,Yes,0,0,0,0,1,0,0,0,1,0
56753,CUS_0x1000,202202,17,"30,625.94","2,706.16",6,27,2,62,1.63,11.00,"1,562.91",29.44,10.25,42.94,176.13,311.54,23.00,Lawyer,Bad,High_spent_Small_value_payments,Yes,0,0,0,0,1,0,0,0,1,1
56754,CUS_0x1000,202203,17,"30,625.94","2,706.16",6,27,2,62,1.63,11.00,"1,562.91",38.29,10.33,42.94,109.06,368.62,28.00,Lawyer,Bad,High_spent_Medium_value_payments,Yes,0,0,0,0,1,0,0,0,1,0
56755,CUS_0x1000,202204,17,"30,625.94","2,706.16",6,27,2,64,1.63,11.00,"1,562.91",32.84,10.42,42.94,87.91,419.77,25.00,Lawyer,Bad,<NA>,Yes,0,0,0,0,1,0,0,0,1,0
56756,CUS_0x1000,202205,17,"30,625.94","2,706.16",6,27,2,67,2.63,11.00,"1,562.91",32.33,10.50,42.94,191.83,305.84,25.00,Lawyer,Bad,Low_spent_Large_value_payments,Yes,0,0,0,0,1,0,0,0,1,0
56757,CUS_0x1000,202206,18,"30,625.94","2,706.16",6,27,2,62,1.63,11.00,"1,562.91",40.08,10.58,42.94,114.80,372.87,23.00,Lawyer,Bad,High_spent_Small_value_payments,Yes,0,0,0,0,1,0,0,0,1,0
56758,CUS_0x1000,202207,18,"30,625.94","2,706.16",6,27,2,62,2.63,11.00,"1,562.91",38.15,10.67,42.94,266.60,251.08,25.00,Lawyer,Bad,Low_spent_Small_value_payments,Yes,0,0,0,0,1,0,0,0,1,0
56759,CUS_0x1000,202208,18,"30,625.94","2,706.16",6,27,2,57,1.63,11.00,"1,562.91",30.08,10.75,42.94,77.31,400.36,26.00,Lawyer,Bad,High_spent_Medium_value_payments,Yes,0,0,0,0,1,0,0,0,1,0


TODO: — Feature Engineering Experiment: Payment_Behaviour

Understand how feature representation affects model performance by comparing:
	•	Combined categorical feature vs
	•	Split features vs
	•	Split + interactiom